# BigMart Sales Prediction

Predict `Item_Outlet_Sales` using regression models, with XGBoost as the primary model.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## 2. Load Dataset

Place the permitted BigMart `Train.csv` file inside `data/` before running this cell.

In [ ]:
DATA_PATH = "data/Train.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "Train.csv was not found. Add the dataset to data/Train.csv and run the notebook again."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


## 3. Initial Data Inspection

In [ ]:
print(df.info())
display(df.describe(include="all").T)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 4. Data Cleaning

In [ ]:
df = df.copy()

# Standardize common categorical inconsistencies when present.
if "Item_Fat_Content" in df.columns:
    df["Item_Fat_Content"] = df["Item_Fat_Content"].replace({
        "LF": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular"
    })

# Fill Item_Weight with the median within Item_Identifier where possible.
if "Item_Weight" in df.columns:
    if "Item_Identifier" in df.columns:
        df["Item_Weight"] = df.groupby("Item_Identifier")["Item_Weight"].transform(
            lambda s: s.fillna(s.median())
        )
    df["Item_Weight"] = df["Item_Weight"].fillna(df["Item_Weight"].median())

print("Missing values after cleaning:\n", df.isnull().sum())


## 5. Exploratory Data Analysis

In [ ]:
if "Item_Outlet_Sales" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df["Item_Outlet_Sales"], kde=True)
    plt.title("Distribution of Item Outlet Sales")
    plt.xlabel("Item Outlet Sales")
    plt.show()

if "Item_MRP" in df.columns and "Item_Outlet_Sales" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x="Item_MRP", y="Item_Outlet_Sales", alpha=0.5)
    plt.title("Item MRP vs Item Outlet Sales")
    plt.show()


## 6. Prepare Features and Target

In [ ]:
TARGET = "Item_Outlet_Sales"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found in the dataset.")

X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 7. Train Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

results = []
predictions = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    predictions[name] = pred

    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
display(results_df)


## 8. XGBoost: Actual vs Predicted

In [ ]:
xgb_pred = predictions["XGBoost"]

plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=xgb_pred, alpha=0.5)
min_val = min(y_test.min(), xgb_pred.min())
max_val = max(y_test.max(), xgb_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("XGBoost: Actual vs Predicted Sales")
plt.show()


## 9. Final Result

Use the `results_df` table above to record the final reproduced metrics in `results/metrics_template.txt` and in the README if required.